# 00 – Gagnaskoðun

Áður en við förum að þjálfa líkön viljum við **kynnast gögnunum**. Í þessu skjali skoðum við:

1. Stærð og uppbyggingu kjósendakönnunar ÍSKOS 2024.
2. Lýsigögn — hvernig allar breytur eru íslenskaðar með þeim merkingum sem `pyreadstat` les beint úr .sav skránni.
3. Útkomubreytuna (flokksval) og hvernig hún er samanborið við niðurstöður alþingiskosninganna 2024.
4. Eyður í gögnunum og einingauppbyggingu könnunar.
5. Mynd af lykilbreytum (lrscale, age, education, ...) sem eru notaðar í líkönunum.
6. Dreifingu **sérhvers dálks** í gagnasafninu (ein mynd á hverja breytu).

**Þetta skjal skrifar ekkert á disk** sem er notað síðar. Hún er til skoðunar, öll hreinsun og skipting fer fram í `01_data_prep.ipynb`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadstat
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "Kjósendakönnun" / "ÍSKOS kjósenda 2024" / "ISKOS2024_med_aukabreytum_20260318.sav"

# Make src/ importable so we can use the readable-name helper
_SRC = str((PROJECT_ROOT / "src").resolve())
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)
from iskos_labels import IskosLabels  # noqa: E402

# Load variable + value labels directly from the .sav file's SPSS metadata
# (metadataonly=True means we don't read the 30MB of data — just the labels).
# feature_groups.json is produced by notebook 01; we pass it if it exists so
# one-hot columns like 'educationflo_80' can be split back into (var, value).
FG_PATH = PROJECT_ROOT / "data" / "processed" / "feature_groups.json"
labels = IskosLabels.from_sav(DATA_PATH, feature_groups_path=FG_PATH if FG_PATH.exists() else None)
print(f"Loaded {len(labels.variable_labels)} variable labels and "
      f"{len(labels.value_labels)} value-labelled variables.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

## 1 — Heildaryfirlit

Hve margir svarendur, hve margar breytur? Hver eru helstu lýsigögnin?

In [ ]:
df_raw, meta = pyreadstat.read_sav(str(DATA_PATH), apply_value_formats=False)
print(f"Stærð gagnanna : {df_raw.shape[0]:,} svarendur × {df_raw.shape[1]:,} breytur")
print(f"Skráarheiti    : {DATA_PATH.name}")
print(f"Skráarstærð    : {DATA_PATH.stat().st_size / 1024 / 1024:.1f} MB")
print()
print("Nokkrar lýsigagnabreytur úr SPSS (þær fyrstu):")
for col in df_raw.columns[:8]:
    print(f"  {col:25s}  {meta.column_names_to_labels.get(col, '')[:80]}")

## 2 — Réttar nafngiftir á breytum

Breyturnar í .sav skránni hafa stutt enska kóðanöfn (t.d. `lrscale`, `govprfm`, `bprteco`). Til þess að myndir og töflur séu lesanlegar þurfum við **íslensk nöfn** sem bæði segja hvað breytan mælir og hvað hvert gildi táknar.

Sem betur fer lifa þessar nafngiftir nú þegar **innbyggðar í .sav skránni** — SPSS geymir bæði spurningartextann (`column_names_to_labels` í `meta`-hlutnum) og kortlagningu frá kóða yfir í íslenskan texta fyrir hvert gildi (`variable_value_labels`). Ekkert þarf að greiða úr docx skrám.

Við notum `IskosLabels`-hjálpartólið (í `src/iskos_labels.py`) til að fá lesanlegt nafn fyrir hvaða breytu (eða one-hot dálk) sem er. Það kallar `pyreadstat.read_sav(..., metadataonly=True)` og byggir úr því dálkmerkingar. Eina viðbótin er handgert `SHORT_QUESTION_LABELS_IS`-orðasafn með 1–3 orða *stuttmerkingu* sem passa á áshausa á myndritum.

In [ ]:
# Sample lookups — picking a couple of variables to demonstrate
demo = ["lrscale", "govprfm", "educationflo", "bprteco", "EUmship", "ecobettwors_1"]

print("Spurningartexti fyrir nokkrar breytur:")
for v in demo:
    print(f"  {v:18s} -> {labels.question(v)[:90]}")

print("\nGildismerkingar fyrir 'lrscale':")
for code_, lab in (labels.value_labels.get("lrscale") or {}).items():
    print(f"  {code_:>4s}  {lab}")

print("\nDæmi um one-hot dálk-nöfn → lesanlegt:")
for col in ["lrscale", "educationflo_80", "elecdist_4", "bprteco_3", "urban_3"]:
    print(f"  {col:22s} -> {labels.readable(col)}")

## 3 — Útkomubreytan: flokksval (`prtvote24`)

Hér skoðum við dreifingu svaranna við spurningunni „kaust þú X?" og berum saman við **opinber úrslit kosninganna 2024**. Hlutföllin þurfa að líkjast hvort öðru til þess að ÍSKOS gögnin endurspegli kjörfylgið.

In [ ]:
# Raw distribution of prtvote24 with party labels from the metadata
prtvote24 = df_raw["prtvote24"]
counts = prtvote24.value_counts(dropna=False).to_frame("n")
counts["pct_of_sample"] = (counts["n"] / counts["n"].sum() * 100).round(1)
counts["label"] = counts.index.map(
    lambda c: labels.value_labels.get("prtvote24", {}).get(
        str(int(c)) if pd.notna(c) and float(c).is_integer() else str(c), str(c)
    )
)
counts.index.name = "kódi"
counts[["label", "n", "pct_of_sample"]].head(20)

In [ ]:
# 2024 official seat-allocation results (Alþingiskosningar 30. nóvember 2024)
# Source: Hagstofan / Landskjörstjórn — used here to sanity-check the survey.
# Keys include the party's one-letter code in parentheses so we can join on the
# letter (see `letter_code()` below). This avoids the accusative-vs-nominative
# trap in the Icelandic labels: the SPSS value labels use the accusative form
# ("Samfylkinguna (S)") while this hand-written table uses the nominative
# ("Samfylkingin (S)"). The letter in parentheses is identical in both.
import re

official_2024 = {
    "Samfylkingin (S)":                     20.8,
    "Sjálfstæðisflokkurinn (D)":            19.4,
    "Viðreisn (C)":                         15.8,
    "Miðflokkurinn (M)":                    12.1,
    "Flokkur fólksins (F)":                 13.8,
    "Framsóknarflokkurinn (B)":              7.8,
    "Sósíalistaflokkur Íslands (J)":         4.0,
    "Píratar (P)":                           3.0,
    "Vinstrihreyfingin grænt framboð (V)":   2.3,
}

# Canonical display name per letter code. The 9 entries below are the
# nominative forms from `official_2024`; L and Y were on the ballot
# in 2024 but are not in the official_2024 table, so we add them by hand.
DISPLAY_NAME_IS = {
    "S": "Samfylkingin",
    "D": "Sjálfstæðisflokkurinn",
    "C": "Viðreisn",
    "M": "Miðflokkurinn",
    "F": "Flokkur fólksins",
    "B": "Framsóknarflokkurinn",
    "J": "Sósíalistaflokkur Íslands",
    "P": "Píratar",
    "V": "Vinstrihreyfingin grænt framboð",
    "L": "Lýðræðisflokkurinn",
    "Y": "Ábyrg framtíð",
}


def letter_code(label: str) -> str | None:
    """Extract the single uppercase letter in parentheses at the end of a label.

    Works for both accusative ("Samfylkinguna (S)") and nominative
    ("Samfylkingin (S)") forms — only the letter matters.
    """
    m = re.search(r"\(([A-Z])\)\s*$", label)
    return m.group(1) if m else None


# In-sample percentages for parties that received any votes (filtered to coded parties only)
party_codes = {1, 2, 3, 4, 6, 8, 10, 11, 12, 13, 14}
in_sample = prtvote24[prtvote24.isin(party_codes)]
in_sample_pct = (in_sample.value_counts(normalize=True) * 100).round(1)

# Re-index BY LETTER CODE extracted from the SPSS value labels
spss_labels = labels.value_labels.get("prtvote24", {})
in_sample_pct.index = [
    letter_code(spss_labels.get(str(int(c)), "")) or f"?{c}"
    for c in in_sample_pct.index
]

# Same transformation on the official table's keys
official_series = pd.Series(official_2024)
official_series.index = [letter_code(k) or k for k in official_series.index]

# Join on letter code (the real party identity), then attach a human label
cmp = pd.DataFrame({
    "ÍSKOS úrtak (%)": in_sample_pct,
    "Opinber úrslit (%)": official_series,
})
cmp["Mismunur (pp)"] = cmp["ÍSKOS úrtak (%)"] - cmp["Opinber úrslit (%)"]
cmp.insert(0, "Flokkur", [DISPLAY_NAME_IS.get(code, code) for code in cmp.index])
cmp.index.name = "Kóði"
cmp = cmp.sort_values("Opinber úrslit (%)", ascending=False, na_position="last").round(1)

print("Samanburður: úrtakshlutfall vs. opinber úrslit (þingflokkar og utanþings flokkar með):")
# Show em-dash in place of NaN only for display — the underlying numeric columns
# are kept intact above so sort_values and arithmetic work.
cmp.fillna("–")

## 4 — Eyður í gögnunum

ÍSKOS notar **einingauppbyggingu** — hver svarandi svarar kjarnaspurningalistanum plús nokkrum einingum, en sleppir öðrum. Niðurstaðan er sú að margar breytur  **vantar því kerfisbundið** hjá hluta úrtaksins, ekki vegna rangra svara heldur vegna þess að ekki var spurt.

Sérstök gildi sem þýða „vantar" eru samkvæmt SPSS staðalnum:

| Kóði | Merking |
|---|---|
| 995 | Ekki spurt (eining ekki sett upp fyrir svaranda) |
| 996 | Á ekki við |
| 997 | Neitar að svara |
| 998 | Veit ekki |
| 999 | Brottfall (skráð) |

Hér skoðum við hvaða breytur vantar oftast (`NaN` í pyreadstat eftir að þessi sérstöku gildi eru meðhöndluð sem `NaN`).

In [ ]:
# Top 25 columns by missing rate
missing = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing_df = missing.head(25).rename("missing_pct").to_frame()
missing_df["lýsing"] = missing_df.index.map(lambda v: labels.short_question(v))
missing_df["spurning (stutt)"] = missing_df.index.map(lambda v: labels.question(v)[:80])
missing_df.round(1)

In [ ]:
# Bar chart of missingness for the modelling-relevant variable groups
modelling_vars = [
    # Social status
    "gender", "age", "educationflo", "elecdist", "urban", "emplstatus",
    "incomecate", "rentown", "maritals", "hhmmb", "childs", "languagecat",
    "church", "rlgatnd",
    # Issue attitudes
    "lrscale", "polintr", "pltrust", "envirimp", "gincdif", "taxreduc",
    "capicntry", "ceqvote", "privhealth", "imueclt", "essfull", "femfight",
    "banks", "immrght", "genconsind", "acrproimp", "drugpol", "gndequ", "EUmship",
    # Government performance
    "govprfm", "ecobettwors_1",
    # Issue ownership
    "bprteco", "bprtimm", "bprteurope", "bprtfamily", "bprtisencit",
    "bprthealth", "bprtenvir", "bprtregpol", "bprtedu", "bprtequal", "bprthouse",
]
miss = (df_raw[modelling_vars].isna().mean() * 100).sort_values()
labels_y = [labels.short_question(v) for v in miss.index]

fig, ax = plt.subplots(figsize=(9, max(8, 0.25 * len(miss))))
ax.barh(labels_y, miss.values, color="#4a90a4")
ax.set_xlabel("Hlutfall vantandi gilda í úrtakinu (%)")
ax.set_title("Eyður í forspárbreytunum (úr öllu úrtakinu)")
ax.set_xlim(0, max(100, miss.max() + 5))
plt.tight_layout()
plt.show()

## 5 — Snögg sýn á lykilbreytur

Stuttar súlumyndir/dreifingar fyrir nokkrar breytur sem koma sterkast fram í SHAP-greiningunni síðar (í þriðja skjalinu). Þetta gefur lesendum tilfinningu fyrir gildum — t.d. „hvar liggur miðjan á hægri-vinstri ásnum?" og „hve mörg hlutfall svarenda er í Reykjavík?"

In [ ]:
key_vars = ["lrscale", "govprfm", "ecobettwors_1", "age", "educationflo", "elecdist"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for ax, v in zip(axes, key_vars):
    s = df_raw[v].dropna()
    short = labels.short_question(v)
    if v in labels.value_labels:
        # Categorical / ordinal — bar chart with value labels
        counts = s.value_counts().sort_index()
        x = [labels.value_label(v, k) for k in counts.index]
        ax.bar(range(len(counts)), counts.values, color="#4a90a4")
        ax.set_xticks(range(len(counts)))
        ax.set_xticklabels(x, rotation=45, ha="right", fontsize=8)
    else:
        # Numeric — histogram
        ax.hist(s, bins=30, color="#4a90a4", edgecolor="white")
    ax.set_title(short)
    ax.set_ylabel("Fjöldi svarenda")
plt.suptitle("Lykilbreytur — dreifing í úrtakinu", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6 — Dreifing allra dálka

Hér undir er **ein mynd fyrir hvern dálk** í gagnasafninu — tilgangurinn er að lesandi geti flett í gegn og fengið tilfinningu fyrir hverri einustu breytu, ekki bara þeim sem við notum í líkönunum. Flokkabreytur fá súluriti (eitt súlustæði per gildi) en tölulegar breytur fá stuðarit (*histogram*).

> **Ath.** Þetta framkallar um 175 myndir í röð og tekur smá stund að birta. Skrollaðu í gegn til að fá góða yfirsýn.

In [ ]:
# Simple loop — one plot per column in df_raw.
# Categorical variables (those with value labels) → bar chart with label ticks.
# Numeric variables                               → histogram.
# We skip columns that are entirely missing. Expect ~175 figures, so scroll.
for col in df_raw.columns:
    s = df_raw[col].dropna()
    if len(s) == 0:
        print(f"(Sleppti {col} — öll gildi vantar)")
        continue

    fig, ax = plt.subplots(figsize=(9, 3.5))
    short = labels.short_question(col)
    full = labels.question(col)

    if col in labels.value_labels:
        # Categorical / ordinal — sort by numeric code so ordering is stable
        counts = s.value_counts().sort_index()
        x_lbls = [labels.value_label(col, k) for k in counts.index]
        ax.bar(range(len(counts)), counts.values, color="#4a90a4")
        ax.set_xticks(range(len(counts)))
        # Truncate long labels so they fit under the bars
        ax.set_xticklabels(
            [lbl if len(lbl) <= 35 else lbl[:32] + "…" for lbl in x_lbls],
            rotation=45, ha="right", fontsize=8,
        )
    else:
        ax.hist(s, bins=30, color="#4a90a4", edgecolor="white")

    # Column code + short label in the title; full question text just below
    ax.set_title(f"{col} – {short}", fontsize=11, loc="left")
    ax.text(
        0.0, 1.12, full[:120],
        transform=ax.transAxes, fontsize=8, color="#666", va="bottom",
    )
    ax.set_ylabel("Fjöldi svarenda")
    plt.tight_layout()
    plt.show()

## 7 — Spurningar sem voru lagðar fyrir megnið af úrtakinu

ÍSKOS er byggð upp í **einingum**: kjarnalistinn fer fyrir alla svarendur en hver eining (t.d. CSES-eining eða ESS-sambærileg) er aðeins spurð af hluta úrtaksins. Þegar spurning er ekki lögð fyrir svaranda skráist sérstök SPSS-kóði (oftast **995 – „Ekki spurt"**). Ef við lesum gögnin með `user_missing=False` (eins og í §1) hverfa þessi gildi í `NaN` og við getum ekki greint „ekki spurt" frá öðrum ástæðum brottfalls.

Hér lesum við .sav skrána aftur með `user_missing=True`, finnum þær breytur sem hafa „Ekki spurt" sem leyfilegan valkost, og birtum þær þar sem **minna en 5%** svarenda fengu þá spurningu ekki lagða fyrir sig. Þetta eru þær spurningar sem voru nær allir í úrtakinu spurðir — þ.e. kjarnaspurningar sem hægt er að treysta á við líkanagerð án þess að þurfa að hugsa sérstaklega um einingarbrot.

In [ ]:
# Re-read the .sav with user_missing=True so that SPSS user-missing codes
# (995 = "Ekki spurt", 996 = "Á ekki við", 997 = "Neitar", 998 = "Veit ekki",
# 999 = "Brottfall") are preserved as numeric values instead of being turned
# into NaN. The default `user_missing=False` (which df_raw uses) conflates
# these with real missingness.
df_codes, _ = pyreadstat.read_sav(
    str(DATA_PATH), apply_value_formats=False, user_missing=True
)


def ekki_spurt_codes_for(var: str) -> list[int]:
    """Return the numeric code(s) that mean 'Ekki spurt' for this variable.
    We discover them by scanning the value-label text rather than assuming 995,
    since different variables may use slightly different conventions."""
    return [
        int(code)
        for code, lbl in labels.value_labels.get(var, {}).items()
        if "ekki spurt" in lbl.lower()
    ]


# For every variable that has "Ekki spurt" as a labelled option, count the
# proportion of respondents whose answer was that code.
rows = []
for col in df_codes.columns:
    codes = ekki_spurt_codes_for(col)
    if not codes:
        continue  # variable doesn't offer "Ekki spurt" as a value
    n = int(df_codes[col].isin(codes).sum())
    rows.append({
        "breyta": col,
        "ekki_spurt_pct": n / len(df_codes) * 100,
        "lýsing": labels.short_question(col),
        "spurning": labels.question(col),
    })

ekki_df = pd.DataFrame(rows).sort_values("ekki_spurt_pct").reset_index(drop=True)
low_ekki = ekki_df[ekki_df["ekki_spurt_pct"] < 99].copy()

print(f"Breytur með 'Ekki spurt' sem valkost           : {len(ekki_df)}")
print(f"Þar af með minna en 5% 'Ekki spurt' svara      : {len(low_ekki)}")
print()
print("Spurningar sem voru lagðar fyrir megnið af úrtakinu (< 5% 'Ekki spurt'):")
print("-" * 100)
for _, row in low_ekki.iterrows():
    q = row["spurning"]
    q_short = q #if len(q) <= 90 else q[:87] + "..."
    print(f"  {row['breyta']:22s}  {row['ekki_spurt_pct']:5.2f}%  {q_short}")

## Samantekt

- ÍSKOS 2024 inniheldur **2.658 svarendur** og 175 breytur.
- Sérhver breyta hefur **opinbert íslenskt heiti** sem lesið er beint úr .sav-skránni með `pyreadstat`. Öll skjöl sem fylgja á eftir nota `IskosLabels`-hjálpartólið og þar með sömu nafngiftir.
- Útkomubreytan endurspeglar opinber úrslit innan við ~5 prósentustig fyrir hvern flokk — gögnin eru hæfileg fyrir spá um flokksval.
- Við höfum skoðað dreifingu sérhvers dálks í §6 og auðkennt í §7 þær spurningar sem nánast allir svarendur fengu lagðar fyrir sig (< 5% „Ekki spurt"). Þetta gefur lesendum tilfinningu fyrir hráum gögnum og gildar forsendur fyrir líkanagerð.

**Næst:** `01_data_prep.ipynb` hreinsar gögnin og býr til þjálfunar- og prófunarsett.